In [ ]:
#| default_exp responses

# Responses server

> Provider-independent Responses resources and streaming over FastLLM


This module serves Responses-format requests through FastLLM. It converts input to shared messages, calls the provider, and returns Responses resources and streaming events. A public response can include multiple provider calls when the host supplies tools to run on the server.

Continuation can use a provider response ID or replay the retained messages. The host decides who may continue a response and how to persist it; `ResponseStore` supplies a default in-memory store.

## Imports

In [ ]:
import asyncio, copy, json, secrets, time
from fastcore.utils import *
from fastcore.funccall import call_func_async, get_schema
from fasttransport.errors import APIError

from fastllm.acomplete import acomplete
from fastllm.streaming import Status
from aidialog.msg_parts import Msg, Text, Thinking, ToolUse, ToolResult, InputImage, InputVideo, InputFile, Completion, MdStr, tool_text, msg2dict, dict2msg, mk_part

In [ ]:
from fastcore.test import *
from cachy import enable_cachy
from fastllm.types import Usage, luna
from fastllm.acomplete import defaults, vendor_doms
from aidialog.msg_parts import InputAudio, ToolResponse, TOOL_RESPONSE_MIME

## Input items

`response_input` accepts a prompt string, Responses input items, or canonical `msg2dict` messages. It returns `(messages, system)`. System and developer text becomes the reusable `system` prompt. Canonical messages preserve provider replay data.

In [ ]:
class ResponsesError(Exception):
    "A request error with Responses-compatible metadata."
    def __init__(self, message, status=400, code='invalid_request_error', param=None):
        super().__init__(message)
        store_attr()

`_msg_parts` restores typed content to canonical parts for the selected backend. Audio uses aidialog's serialized `InputAudio` fields; it is a facade extension, not native OpenAI Responses input. A part retains its `cache_control`. Unsupported content types raise `ResponsesError` with `param='input'`.

In [ ]:
def _msg_parts(content):
    if isinstance(content, str): return [Text(content)]
    parts = []
    for o in content:
        typ,cc = o.get('type'),o.get('cache_control')
        if typ in ('input_text', 'output_text', 'text'): parts.append(Text(o.get('text', ''), cache_control=cc))
        elif typ == 'input_image': parts.append(InputImage(o.get('image_url'), cache_control=cc))
        elif typ == 'input_video': parts.append(InputVideo(o.get('video_url'), cache_control=cc))
        elif typ == 'input_file': parts.append(InputFile(o.get('file_url') or o.get('file_data'), cache_control=cc))
        elif typ == 'input_audio': parts.append(mk_part(**o))
        else: raise ResponsesError(f'Unsupported message content type: {typ}', param='input')
    return parts

In [ ]:
text_block = dict(type='input_text', text='Describe this image.')
image_block = dict(type='input_image', image_url='data:image/png;base64,AAAA', cache_control=dict(type='ephemeral'))
parts = _msg_parts([text_block, image_block])
test_eq([type(o) for o in parts], [Text, InputImage])
test_eq([o.cache_control for o in parts], [None, dict(type='ephemeral')])
parts

[Text(raw=None, cache_control=None, text='Describe this image.', citations=None),
 InputImage(raw=None, cache_control={'type': 'ephemeral'}, text='data:image/png;base64,AAAA', mime=None)]

Multiple system and developer items contribute instruction text, separated by blank lines. User messages keep their typed content parts.

In [ ]:
def _text_content(content):
    if isinstance(content, str): return content
    return ''.join(o.get('text', '') for o in content if o.get('type') in ('input_text', 'output_text', 'text'))

def response_input(inp):
    "Convert Responses input items to canonical messages and instructions."
    if isinstance(inp, str): return [Msg('user', [Text(inp)])], ''
    if not isinstance(inp, list): raise ResponsesError('input must be a string or array', param='input')
    msgs,systems,pending_tools = [],[],[]
    def flush_tools():
        if pending_tools:
            msgs.append(Msg('tool', pending_tools.copy()))
            pending_tools.clear()
    for item in inp:
        typ = item.get('type')
        if 'raw_api' in item:
            flush_tools()
            msgs.append(dict2msg(item))
        elif typ in ('message', 'easy_input_message') or 'role' in item:
            flush_tools()
            role = item.get('role', 'user')
            if role in ('system', 'developer'): systems.append(_text_content(item.get('content', '')))
            else: msgs.append(Msg(role, _msg_parts(item.get('content', ''))))
        elif typ == 'function_call':
            flush_tools()
            args = item.get('arguments') or {}
            if isinstance(args, str):
                try: args = json.loads(args)
                except json.JSONDecodeError: pass
            call = ToolUse(id=item.get('call_id'), name=item.get('name'), arguments=args)
            if msgs and msgs[-1].role == 'assistant': msgs[-1].content.append(call)
            else: msgs.append(Msg('assistant', [call]))
        elif typ == 'function_call_output':
            out = item.get('output', '')
            pending_tools.append(ToolResult(id=item.get('call_id'), name=item.get('name', ''),
                text=out if isinstance(out, str) else _msg_parts(out)))
        elif typ == 'reasoning': continue
        else: raise ResponsesError(f'Unsupported input item type: {typ}', param='input')
    flush_tools()
    return msgs,'\n\n'.join(filter(None, systems))

In [ ]:
input_items = [dict(role='developer', content='Answer briefly.'),
    dict(role='user', content=[dict(type='input_text', text='What is shown?')])]
input_msgs,instructions = response_input(input_items)
test_eq((instructions, input_msgs[0].text), ('Answer briefly.', 'What is shown?'))
dict(instructions=instructions, message=input_msgs[0])

{'instructions': 'Answer briefly.',
 'message': Msg(role='user', content=[Text(raw=None, cache_control=None, text='What is shown?', citations=None)], raw=None, raw_api=None)}

A parallel tool batch must stay together. `response_input` appends function calls to the preceding assistant message, then groups adjacent function outputs into one tool message. The next provider sees all the calls followed by all their results.

`response_input` skips replayed `reasoning` items. No provider accepts the summary text in them as its own reasoning.


In [ ]:
batch_items = [dict(type='reasoning', id='rs_a', summary=[dict(type='summary_text', text='Two sums.')]),
    dict(type='message', role='assistant', content=[dict(type='output_text', text='Computing both.')]),
    dict(type='function_call', call_id='call_a', name='py', arguments='{"code":"2+2"}'),
    dict(type='function_call', call_id='call_b', name='py', arguments='{"code":"3*3"}'),
    dict(type='function_call_output', call_id='call_a', output='4'), dict(type='function_call_output', call_id='call_b', output='9')]
batch_msgs,system = response_input(batch_items)
test_eq(system, '')
test_eq([m.role for m in batch_msgs], ['assistant', 'tool'])
test_eq([type(p).__name__ for p in batch_msgs[0].content], ['Text', 'ToolUse', 'ToolUse'])
test_eq([o.id for o in batch_msgs[1].content], ['call_a', 'call_b'])
batch_msgs[0]


**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

Computing both.

::: details

- raw: `None`
- citations: `None`

:::

🔧 **py**(`{'code': '2+2'}`)

::: details

- id: `call_a`
- server: `False`
- raw: `None`

:::

🔧 **py**(`{'code': '3*3'}`)

::: details

- id: `call_b`
- server: `False`
- raw: `None`

:::

</contents>

Tool outputs can contain typed arrays as well as strings. Here the image becomes an `InputImage` in the tool result. A provider adapter can then encode it in its own supported image format.

In [ ]:
img = 'data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP8z4AATAxIAAB2+AEAqfB/ugAAAABJRU5ErkJggg=='
media_out = [dict(type='function_call_output', call_id='call_c',
    output=[dict(type='input_text', text='the chart:'), dict(type='input_image', image_url=img)])]
mmsgs,_ = response_input(media_out)
parts = mmsgs[0].content[0].text
test_eq([type(p).__name__ for p in parts], ['Text', 'InputImage'])
test_eq(parts[1].text, img)
mmsgs[0]
clip = InputAudio('data:audio/wav;base64,UklGRg==', mime='audio/wav', cache_control=dict(type='ephemeral'))
clip_out = dict(type='function_call_output', call_id='clip', output=ToolResponse(clip)._repr_mimebundle_()[TOOL_RESPONSE_MIME])
clip_msgs,_ = response_input([clip_out])
test_eq(clip_msgs[0].content[0].text, [clip])
test_eq(response_input([dict(role='user', content=clip_out['output'])])[0][0].content, [clip])
test_eq(type(_msg_parts([dict(type='input_video', video_url='https://example.com/clip.mp4')])[0]).__name__, 'InputVideo')

## Response resources

`normalize_call_ids` supplies missing client call IDs and rewrites any IDs that collide with `used` history or earlier calls in the same response, keeping `Msg.raw` aligned so later tool results match both canonical and raw replay data.

In [ ]:
def _new_id(prefix): return f'{prefix}_{secrets.token_hex(16)}'

def _history_call_ids(history):
    return {p.id for m in (history or ()) for p in m.content if isinstance(p, ToolUse) and not p.server and p.id}

def _raw_tool_entries(raw, raw_api):
    if raw_api == 'gemini' and isinstance(raw, dict): return [(p['functionCall'], 'id') for p in raw.get('parts') or [] if 'functionCall' in p]
    if raw_api == 'anthropic' and isinstance(raw, dict): return [(b, 'id') for b in raw.get('content') or [] if b.get('type') == 'tool_use']
    if raw_api == 'openai' and isinstance(raw, list): return [(it, 'call_id') for it in raw if it.get('type') == 'function_call']
    if raw_api == 'openai_chat' and isinstance(raw, dict): return [(it, 'id') for it in raw.get('tool_calls') or []]
    return []

def _sync_raw_call_ids(msg):
    "Keep `msg.raw` tool-call IDs aligned with non-server `ToolUse` parts in `msg.content`."
    if not msg.raw or not msg.raw_api: return
    tcs = [p for p in msg.content if isinstance(p, ToolUse) and not p.server]
    entries = _raw_tool_entries(msg.raw, msg.raw_api)
    if not tcs or not any(d.get(k) != tc.id for (d,k),tc in zip(entries, tcs)): return
    raw = copy.deepcopy(msg.raw)
    for (d,k),tc in zip(_raw_tool_entries(raw, msg.raw_api), tcs): d[k] = tc.id
    msg.raw = raw

def normalize_call_ids(comp, used=None):
    "Ensure every non-server tool call has a unique Responses call id across `used` and `comp`, keeping `comp.message.raw` aligned."
    seen = set(used or ())
    for tc in comp.tool_calls:
        if tc.server: continue
        if not tc.id or tc.id in seen: tc.id = _new_id('call')
        seen.add(tc.id)
    _sync_raw_call_ids(comp.message)
    return comp

`response_output` turns each Thinking part into a `reasoning` item. Each run of text parts becomes one message item, with their citations as annotations. A tool call ends the run, and text after it starts a new message item, in the order the stream sent it. Client function calls become their own items.

A completed server-side call uses the `server_tool_call` extension. This includes both host tools and provider tools such as web search. The item carries `arguments` and the result in `output`, with `md: true` for an `MdStr` result. Clients can display the work without treating it as a request to run a tool.


In [ ]:
def _arguments(args): return args if isinstance(args, str) else json.dumps(args or {}, separators=(',', ':'))
def _is_srv(part): return isinstance(part, ToolUse) and part.server

def server_call_item(part):
    "The output item for a call the server ran, whether a host tool or the provider's own: a `server_tool_call` carrying its `arguments` and result `output`, marked `md` when that output is markdown"
    item = dict(type='server_tool_call', name=part.name, arguments=_arguments(part.arguments), output=part.text or '')
    if isinstance(part.text, MdStr): item['md'] = True
    part._response_item_id = getattr(part, '_response_item_id', None) or _new_id('srv')
    return dict(id=part._response_item_id, status='completed', **item)

def _reasoning_item(rid, text=None):
    "A `reasoning` output item, with `text` as its one summary part"
    return dict(id=rid, type='reasoning', summary=[] if text is None else [dict(type='summary_text', text=text)])

def response_output(comp):
    "Convert a Completion to Responses output items, with one message item per run of text between tool calls."
    output,msg = [],None
    for part in comp.message.content:
        if isinstance(part, Text):
            if msg is None:
                content = dict(type='output_text', text=part.text or '', annotations=list(part.citations or []))
                msg = dict(id=getattr(part, '_response_item_id', None) or _new_id('msg'), type='message', status='completed',
                    role='assistant', content=[content])
                output.append(msg)
            else:
                msg['content'][0]['text'] += part.text or ''
                msg['content'][0]['annotations'].extend(part.citations or [])
        elif isinstance(part, Thinking):
            output.append(_reasoning_item(getattr(part, '_response_item_id', None) or _new_id('rs'), part.text or ''))
        elif isinstance(part, ToolUse):
            msg = None
            if part.server: output.append(server_call_item(part))
            else: output.append(dict(id=getattr(part, '_response_item_id', None) or _new_id('fc'), type='function_call', status='completed',
                call_id=part.id, name=part.name, arguments=_arguments(part.arguments)))
    return output

In [ ]:
sample_usage = dict2obj(dict(prompt_tokens=12, completion_tokens=4, total_tokens=16,
    cached_tokens=3, cache_creation_tokens=2, reasoning_tokens=1))
sample_parts = [Thinking('Adding.'), Text('Done.'), ToolUse(name='web_search', arguments={'query':'2+2'}, server=True, text=MdStr('Calculator: https://example.com/calc')),
    Text(' Checked.'), ToolUse(name='py', arguments={'code':'2+2'})]
sample_comp = Completion('demo', Msg('assistant', sample_parts), 'stop', sample_usage)
normalize_call_ids(sample_comp)
sample_output = response_output(sample_comp)
test_eq([o['type'] for o in sample_output], ['reasoning', 'message', 'server_tool_call', 'message', 'function_call'])
test_eq(sample_output[2].get('md'), True)
dup_raw = dict(role='model', parts=[dict(functionCall=dict(id='call_dup', name='py', args=dict(code='1')), thoughtSignature='sig1'),
    dict(functionCall=dict(id='call_dup', name='py', args=dict(code='2')))])
dup_comp = Completion('demo', Msg('assistant', [ToolUse(id='call_dup', name='py', arguments=dict(code='1')), ToolUse(id='call_dup', name='py', arguments=dict(code='2'))], raw=dup_raw, raw_api='gemini'), 'tool_calls')
normalize_call_ids(dup_comp, used={'call_dup'})
dup_ids = [t.id for t in dup_comp.tool_calls]
test_eq((len(set(dup_ids)), 'call_dup' in dup_ids, [p['functionCall']['id'] for p in dup_comp.message.raw['parts']], dup_comp.message.raw['parts'][0]['thoughtSignature']),
    (2, False, dup_ids, 'sig1'))
sample_output

[{'id': 'rs_ac0b8ad062955ec71c9bb03df48e7045',
  'type': 'reasoning',
  'summary': [{'type': 'summary_text', 'text': 'Adding.'}]},
 {'id': 'msg_c5abdb7d135b2445bb25213840c0fbbd',
  'type': 'message',
  'status': 'completed',
  'role': 'assistant',
  'content': [{'type': 'output_text', 'text': 'Done.', 'annotations': []}]},
 {'id': 'srv_df9d36904a509acb990f3b8b0f569752',
  'status': 'completed',
  'type': 'server_tool_call',
  'name': 'web_search',
  'arguments': '{"query":"2+2"}',
  'output': 'Calculator: https://example.com/calc',
  'md': True},
 {'id': 'msg_d739fba0337986cb1c7d16ac53621524',
  'type': 'message',
  'status': 'completed',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': ' Checked.',
    'annotations': []}]},
 {'id': 'fc_37c2db578a36bda2e22164fb7281c875',
  'type': 'function_call',
  'status': 'completed',
  'call_id': 'call_ff5bcf1505287b3fbda0f5847c904d9d',
  'name': 'py',
  'arguments': '{"code":"2+2"}'}]

`response_usage` renames the shared token counts to Responses fields. Input details retain cached tokens and cache writes; output details retain reasoning tokens.

In [ ]:
def response_usage(usage):
    "Convert normalized usage to Responses usage fields."
    if usage is None: return None
    result = dict(input_tokens=usage.prompt_tokens, output_tokens=usage.completion_tokens, total_tokens=usage.total_tokens)
    result['input_tokens_details'] = {'cached_tokens':usage.cached_tokens, 'cache_write_tokens':usage.cache_creation_tokens}
    result['output_tokens_details'] = {'reasoning_tokens':usage.reasoning_tokens}
    return result

In [ ]:
sample_response_usage = response_usage(sample_usage)
test_eq(sample_response_usage['input_tokens_details'], dict(cached_tokens=3, cache_write_tokens=2))
sample_response_usage

{'input_tokens': 12,
 'output_tokens': 4,
 'total_tokens': 16,
 'input_tokens_details': {'cached_tokens': 3, 'cache_write_tokens': 2},
 'output_tokens_details': {'reasoning_tokens': 1}}

`_response_base` supplies the fields shared by pending and terminal resources: identity, timestamps, model, instructions, and request controls. Both resource forms should echo the same settings.

In [ ]:
def _response_base(rid, body, previous_response_id=None, created_at=None):
    result = dict(id=rid, object='response', created_at=int(created_at or time.time()), instructions=body.get('instructions'), model=body['model'])
    result.update(parallel_tool_calls=body.get('parallel_tool_calls', True), previous_response_id=previous_response_id,
        reasoning=body.get('reasoning'), store=body.get('store', True), prompt_cache_key=body.get('prompt_cache_key'), temperature=body.get('temperature'))
    result.update(text=body.get('text', {'format':{'type':'text'}}), tool_choice=body.get('tool_choice', 'auto'),
        tools=body.get('tools', []), top_p=body.get('top_p'), metadata=body.get('metadata', {}))
    return result

In [ ]:
resource_body = dict(model='demo/model', input='Hello', parallel_tool_calls=True)
resource_base = _response_base('resp_demo', resource_body, created_at=123)
test_eq((resource_base['id'], resource_base['created_at']), ('resp_demo', 123))
resource_base

{'id': 'resp_demo',
 'object': 'response',
 'created_at': 123,
 'instructions': None,
 'model': 'demo/model',
 'parallel_tool_calls': True,
 'previous_response_id': None,
 'reasoning': None,
 'store': True,
 'prompt_cache_key': None,
 'temperature': None,
 'text': {'format': {'type': 'text'}},
 'tool_choice': 'auto',
 'tools': [],
 'top_p': None,
 'metadata': {}}

`response_object` adds output items, usage, and the final status to those shared fields. A stop at the output-length limit gives `status='incomplete'` with reason `max_output_tokens`. A stop by the provider's content filter gives reason `content_filter`. The resource keeps the same shape. `output_text` joins the text in its message items.

In [ ]:
_incomplete = dict(length='max_output_tokens', content_filter='content_filter')

def response_object(rid, body, comp, previous_response_id=None, created_at=None):
    "Build a completed Responses resource from a Completion."
    reason = _incomplete.get(comp.finish_reason)
    output = response_output(comp)
    result = _response_base(rid, body, previous_response_id, created_at)
    result.update(completed_at=int(time.time()), status='incomplete' if reason else 'completed', error=None,
        incomplete_details={'reason':reason} if reason else None, output=output, usage=response_usage(comp.usage))
    result['output_text'] = ''.join(c['text'] for o in output if o['type']=='message' for c in o['content'] if c['type']=='output_text')
    return result

In [ ]:
sample_response = response_object('resp_demo', resource_body, sample_comp, created_at=123)
test_eq((sample_response['status'], sample_response['output_text']), ('completed', 'Done. Checked.'))
filtered = response_object('resp_demo', resource_body, Completion('demo', Msg('assistant', [Text('Partial')]), 'content_filter', sample_usage))
test_eq((filtered['status'], filtered['incomplete_details']), ('incomplete', {'reason': 'content_filter'}))
dict(status=sample_response['status'], output=sample_response['output'], usage=sample_response['usage'])

{'status': 'completed',
 'output': [{'id': 'rs_0b5b6996e0802f8b1f1670db0a1a6bd3',
   'type': 'reasoning',
   'summary': [{'type': 'summary_text', 'text': 'Adding.'}]},
  {'id': 'msg_499e0b01596caddb8953166b80a435a7',
   'type': 'message',
   'status': 'completed',
   'role': 'assistant',
   'content': [{'type': 'output_text', 'text': 'Done.', 'annotations': []}]},
  {'id': 'srv_df9d36904a509acb990f3b8b0f569752',
   'status': 'completed',
   'type': 'server_tool_call',
   'name': 'web_search',
   'arguments': '{"query":"2+2"}',
   'output': 'Calculator: https://example.com/calc',
   'md': True},
  {'id': 'msg_23f71183b73fa3ff58a1cf159cdf6e1c',
   'type': 'message',
   'status': 'completed',
   'role': 'assistant',
   'content': [{'type': 'output_text',
     'text': ' Checked.',
     'annotations': []}]},
  {'id': 'fc_9d335c6e9c8ff5a2caf5084e741e2d79',
   'type': 'function_call',
   'status': 'completed',
   'call_id': 'call_ff5bcf1505287b3fbda0f5847c904d9d',
   'name': 'py',
   'argumen

## One Responses turn

`ResponseState` holds the public ID, model, full message history, prompt cache key, system prompt, and optional provider response ID. The host can store it under its own ownership and expiry rules. A provider ID lets the next call send just the new messages; without one, it needs the retained history and system prompt.

In [ ]:
class ResponseState(BasicRepr):
    "Canonical history and optional provider continuation metadata."
    def __init__(self, id, model, history, provider_response_id=None, prompt_cache_key=None, system=''): store_attr()

In [ ]:
base_body = dict(model=f'openai/{luna}', input='Use py.', tools=[])
prior_msg = Msg('user', [Text('Earlier context')])
replay_state = ResponseState('resp_replay', base_body['model'], (prior_msg,))
replay_state

ResponseState(id='resp_replay', model='openai/gpt-6-luna', history=(Msg(role='user', content=[Text(raw=None, cache_control=None, text='Earlier context', citations=None)], raw=None, raw_api=None),), provider_response_id=None, prompt_cache_key=None, system='')

`to_dict` and `from_dict` let a host serialize the state, including its prompt cache key and each message's `raw` replay data.

In [ ]:
@patch
def to_dict(self:ResponseState):
    "This state as a JSON-ready dict"
    return dict(id=self.id, model=self.model, history=[msg2dict(m) for m in self.history],
        provider_response_id=self.provider_response_id, prompt_cache_key=self.prompt_cache_key, system=self.system)

@patch(cls_method=True)
def from_dict(cls:ResponseState, d):
    "The state `to_dict` produced `d` from"
    return cls(d['id'], d['model'], tuple(dict2msg(m) for m in d['history']), d['provider_response_id'], d['prompt_cache_key'], d.get('system', ''))

In [ ]:
test_eq(ResponseState.from_dict(json.loads(json.dumps(replay_state.to_dict()))).history, replay_state.history)
replay_state.to_dict()

{'id': 'resp_replay',
 'model': 'openai/gpt-6-luna',
 'history': [{'role': 'user',
   'content': [{'type': <PartType.text: 'text'>,
     'raw': None,
     'cache_control': None,
     'text': 'Earlier context',
     'citations': None}],
   'raw': None,
   'raw_api': None}],
 'provider_response_id': None,
 'prompt_cache_key': None,
 'system': ''}

`ResponseStore` keeps a state under its public ID for `ttl` seconds, with any metadata the host supplies. For example, `owner` can identify the user allowed to continue it. The host must enforce that policy.

The default store uses memory in the current process. To share states between processes, override `dump` and `load`. They exchange JSON-ready dicts; a Redis implementation could use `SET ... EX` and `GET`.

In [ ]:
class ResponseStore:
    "Continuation states kept for `ttl` seconds; override `dump` and `load` to keep them outside this process"
    def __init__(self, ttl=3600): self.ttl,self.items = ttl,{}

    async def dump(self, rid, data):
        "Keep JSON-ready `data` under `rid` for `ttl` seconds"
        now = time.time()
        self.items = {k:v for k,v in self.items.items() if v[0] > now}
        self.items[rid] = (now+self.ttl, data)

    async def load(self, rid):
        "The data kept under `rid`, or `None` once it has expired"
        exp,data = self.items.get(rid, (0, None))
        return data if exp > time.time() else None

    async def put(self, state, **meta):
        "Store `state` with the host's `meta` about it"
        await self.dump(state.id, dict(state=state.to_dict(), **meta))

    async def get(self, rid):
        "The `(state, meta)` stored under `rid`, or `None`"
        data = await self.load(rid)
        if data is None: return None
        return ResponseState.from_dict(data['state']),{k:v for k,v in data.items() if k != 'state'}

In [ ]:
store = ResponseStore(ttl=60)
await store.put(replay_state, owner='alice')
state,meta = await store.get('resp_replay')
test_eq((state.history, meta), (replay_state.history, dict(owner='alice')))
test_eq((await store.get('resp_replay'))[1], meta)
test_is(await store.get('resp_missing'), None)
await ResponseStore(ttl=0).put(replay_state)
test_is(await ResponseStore(ttl=0).get('resp_replay'), None)
meta

{'owner': 'alice'}

`JsonStore` demonstrates a serialized backend without a database. It keeps JSON strings in a dict and reconstructs the state on load, as a Redis or database adapter could.

In [ ]:
class JsonStore(ResponseStore):
    def __init__(self): self.strs = {}
    async def dump(self, rid, data): self.strs[rid] = json.dumps(data)
    async def load(self, rid): return json.loads(self.strs[rid]) if rid in self.strs else None

jstore = JsonStore()
await jstore.put(replay_state, owner='alice')
state,meta = await jstore.get('resp_replay')
test_eq(state.history, replay_state.history)
jstore.strs['resp_replay']

'{"state": {"id": "resp_replay", "model": "openai/gpt-6-luna", "history": [{"role": "user", "content": [{"type": "text", "raw": null, "cache_control": null, "text": "Earlier context", "citations": null}], "raw": null, "raw_api": null}], "provider_response_id": null, "prompt_cache_key": null, "system": ""}, "owner": "alice"}'

`ResponseTurn` prepares one provider call. Its `history` keeps the full conversation for the next state. Its `provider_messages` contains the input that this particular call needs.

In [ ]:
class ResponseTurn(BasicRepr):
    "A prepared Responses request for one provider call."
    def __init__(self, id, created_at, body, history, provider_messages, system='', previous=None, server_tools=None, call_kwargs=None):
        store_attr()
        if server_tools is None: self.server_tools = {}
        if call_kwargs is None: self.call_kwargs = {}

    @property
    def provider_previous_id(self): return self.previous.provider_response_id if self.previous else None


`prepare` checks the request, converts new input, carries `developer` and `system` input items forward as `turn.system` (retaining `previous.system` across continuations), and chooses full replay or provider-ID continuation. Top-level `body['instructions']` combines with `turn.system` in `provider_kwargs` without carrying over in `ResponseState`. It also restores missing tool-result names from matching calls in history.

The host can supply `server_tools` to run inside the request. The provider sees their schemas beside the client's tools. Extra keyword arguments become `call_kwargs` and follow every provider call in the turn. A multi-user host can pass the current user's `oauth_token` this way.

In [ ]:
class AsyncResponses:
    "Run individual Responses turns over FastLLM."
    def __init__(self): self.tasks = set()   # served turns still running
    def prepare(self, body, previous=None, server_tools=None, **kwargs):
        if not isinstance(body, dict): raise ResponsesError('Request body must be an object')
        model = body.get('model')
        if not model: raise ResponsesError('model is required', param='model')
        if 'input' not in body: raise ResponsesError('input is required', param='input')
        if previous and previous.model != model: raise ResponsesError('model must match the previous response', param='model')
        new_msgs,input_system = response_input(body['input'])
        history = (*previous.history, *new_msgs) if previous else tuple(new_msgs)
        names = {p.id: p.name for m in history for p in m.content if isinstance(p, ToolUse)}
        for m in new_msgs:
            for p in m.content:
                if isinstance(p, ToolResult) and not p.name: p.name = names.get(p.id, '')
        system = '\n\n'.join(filter(None, [previous.system if previous else '', input_system]))
        provider_messages = new_msgs if previous and previous.provider_response_id else list(history)
        key = body.get('prompt_cache_key') or (previous.prompt_cache_key if previous else None) or secrets.token_hex(16)
        body = body | dict(prompt_cache_key=key)
        return ResponseTurn(_new_id('resp'), int(time.time()), body, history, provider_messages, system, previous,
            {f.__name__: f for f in server_tools or []}, kwargs)


These examples compare the prepared message lists with and without a provider continuation ID. A route handler doesn't need to make that choice itself.

In [ ]:
provider_state = ResponseState('resp_provider', base_body['model'], (prior_msg,), 'upstream_123')
responses = AsyncResponses()
replay_turn = responses.prepare(base_body, replay_state)
provider_turn = responses.prepare(base_body, provider_state)
test_eq(len(replay_turn.provider_messages), 2)
test_eq(len(provider_turn.provider_messages), 1)
(provider_turn.provider_previous_id, [m.text for m in replay_turn.provider_messages])
user_turn = responses.prepare(base_body, api_key='per-user-credential')
test_eq(user_turn.call_kwargs, dict(api_key='per-user-credential'))

`provider_kwargs` supplies the provider options used by `call`. A host can use the same options when inspecting or counting input.

`call` starts a provider stream and waits for its first chunk. A host can call it before sending HTTP headers so an initial provider rejection becomes an HTTP error, not an error halfway through the response.

It forwards the Responses controls that FastLLM supports, combining top-level `body['instructions']` and `turn.system` (from `developer` and `system` input items) into `system`. The `cache_idxs` and `ttl` extensions let a host mark a stable prompt prefix. Anthropic applies those breakpoints; other adapters can ignore them. `web_search_options` requests provider-native search through the appropriate adapter.

In [ ]:
@patch
def provider_kwargs(self:AsyncResponses, turn):
    "Provider request options shared by calling and input inspection"
    body = turn.body
    reasoning = body.get('reasoning') or {}
    tools = list(body.get('tools') or []) + [dict(type='function', **get_schema(f, pname='parameters')) for f in turn.server_tools.values()]
    system = '\n\n'.join(filter(None, [body.get('instructions'), turn.system])) or None
    kwargs = dict(model=body['model'], previous_response_id=turn.provider_previous_id, system=system, tools=tools or None)
    kwargs.update(prompt_cache_key=body['prompt_cache_key'], tool_choice=body.get('tool_choice'), parallel_tool_calls=body.get('parallel_tool_calls', True),
        reasoning_effort=reasoning.get('effort'), max_tokens=body.get('max_output_tokens'), temperature=body.get('temperature'),
        cache_idxs=body.get('cache_idxs'), ttl=body.get('ttl'), web_search_options=body.get('web_search_options'))
    return kwargs

@patch
async def call(self:AsyncResponses, turn):
    "Start a provider stream, waiting for its first chunk before returning"
    stream = await acomplete(turn.provider_messages, stream=True, **self.provider_kwargs(turn), **turn.call_kwargs)
    first = await anext(stream)
    async def _rest():
        yield first
        async for o in stream: yield o
    return _rest()


`state` appends the assistant message and any server-tool results to the request history. It also records the provider's continuation ID, when available. The host can persist this state for the next public request.

In [ ]:
@patch
def state(self:AsyncResponses, turn, comp, results=()):
    "Build the next continuation state; `results` are server tool results answering `comp`'s calls."
    normalize_call_ids(comp, _history_call_ids(turn.history))
    history = (*turn.history, comp.message, *([Msg('tool', list(results))] if results else []))
    return ResponseState(turn.id, turn.body['model'], history, comp.response_id, turn.body['prompt_cache_key'], turn.system)


`prepare` uses the request's `prompt_cache_key`, otherwise the previous state's key, otherwise a new key. The key survives serialization and server-tool rounds. Callers that replay full history without a previous state must supply their conversation's key on each request.

In [ ]:
keyed_body = base_body | dict(prompt_cache_key='solveit:dialog-42', instructions='Now.',
    input=[dict(role='developer', content='Be brief.'), dict(role='user', content='Use py.')])
keyed_turn = responses.prepare(keyed_body)
test_eq(responses.provider_kwargs(keyed_turn)['system'], 'Now.\n\nBe brief.')
keyed_state = responses.state(keyed_turn, sample_comp)
restored_state = ResponseState.from_dict(json.loads(json.dumps(keyed_state.to_dict())))
next_turn = responses.prepare(base_body, restored_state)
test_eq((next_turn.body['prompt_cache_key'], next_turn.system, responses.provider_kwargs(next_turn)['system']),
    (keyed_turn.body['prompt_cache_key'], 'Be brief.', 'Be brief.'))
next_turn.body['prompt_cache_key']

'solveit:dialog-42'

The host runs functions in `server_tools`; the client runs its own tools. `server_results` matches calls by name and executes only the host's functions. It returns `ToolResult` parts.

`server_turn` submits those results for another provider call while retaining the same public response ID. Clients see completed host calls as `server_tool_call` items, just like provider-run search.


In [ ]:
@patch
async def server_results(self:AsyncResponses, turn, comp):
    "Run the server tool calls in `comp`, returning their results"
    normalize_call_ids(comp, _history_call_ids(turn.history))
    res = []
    for tc in comp.tool_calls:
        if tc.name not in turn.server_tools: continue
        out = await call_func_async(tc.name, tc.arguments, ns=turn.server_tools, raise_on_err=False)
        res.append(ToolResult(id=tc.id, name=tc.name, arguments=tc.arguments, text=tool_text(out)))
    return res

@patch
def server_turn(self:AsyncResponses, turn, comp, results):
    "The follow-up provider turn answering server tool `results`, within the same public response"
    outputs = [dict(type='function_call_output', call_id=r.id, output=r.text) for r in results]
    nxt = self.prepare(turn.body | dict(input=outputs), self.state(turn, comp), turn.server_tools.values(), **turn.call_kwargs)
    nxt.id = turn.id
    return nxt

In [ ]:
def lookup(word:str)->str: # The glossary entry
    "Look up `word` in the project glossary"
    return {'gorp': 'a trail mix of nuts and dried fruit'}.get(word, 'not in the glossary')
lookup_turn = responses.prepare(base_body, server_tools=[lookup])
lookup_comp = Completion('demo', Msg('assistant', [ToolUse(id='call_1', name='lookup', arguments=dict(word='gorp')), ToolUse(id='call_2', name='py', arguments=dict(code='1'))]), 'tool_calls')
lookup_results = await responses.server_results(lookup_turn, lookup_comp)
test_eq([(r.name, r.text) for r in lookup_results], [('lookup', 'a trail mix of nuts and dried fruit')])
follow = responses.server_turn(lookup_turn, lookup_comp, lookup_results)
test_eq((follow.id, [m.role for m in follow.history]), (lookup_turn.id, ['user', 'assistant', 'tool']))
replayed,_ = response_input(json.loads(json.dumps([msg2dict(m) for m in follow.history])))
test_eq(replayed, list(follow.history))
follow.provider_messages[-1]

**Msg**

- role: `tool`

<contents>

**ToolResult** (`tool_result`)

a trail mix of nuts and dried fruit

::: details

- raw: `None`
- id: `call_1`
- name: `lookup`
- arguments: `{}`
- server: `False`
- md: `False`
- full: `False`

:::

</contents>

## Event streams

FastLLM calls stream internally. The helpers here turn their parts into Responses SSE frames; they don't decide storage, ownership, or billing policy.

In [ ]:
def response_event(event_type, sequence_number, **kwargs):
    "Encode one Responses SSE frame."
    data = {'type':event_type, **kwargs, 'sequence_number':sequence_number}
    return f"event: {event_type}\ndata: {json.dumps(data, separators=(',', ':'))}\n\n"

In [ ]:
#| hide
def _event_types(frames): return [json.loads(frame.split('data: ', 1)[1])['type'] for frame in frames]

In [ ]:
frame = response_event('response.output_text.delta', 0, delta='Hello')
test_eq(_event_types([frame]), ['response.output_text.delta'])
frame

'event: response.output_text.delta\ndata: {"type":"response.output_text.delta","delta":"Hello","sequence_number":0}\n\n'

`pending_response` starts with `status='in_progress'`, empty output, and no usage. It echoes the request and the previous public response ID.

In [ ]:
def pending_response(turn):
    "Build the initial in-progress Responses resource."
    previous_id = turn.previous.id if turn.previous else None
    result = _response_base(turn.id, turn.body, previous_id, turn.created_at)
    result.update(status='in_progress', error=None, incomplete_details=None, output=[], usage=None)
    return result

In [ ]:
pending = pending_response(provider_turn)
test_eq((pending['status'], pending['previous_response_id']), ('in_progress', 'resp_provider'))
pending

{'id': 'resp_573b2a7d8216e7df2408ef69c7701386',
 'object': 'response',
 'created_at': 1795401222,
 'instructions': None,
 'model': 'openai/gpt-6-luna',
 'parallel_tool_calls': True,
 'previous_response_id': 'resp_provider',
 'reasoning': None,
 'store': True,
 'prompt_cache_key': 'ba1d991dc6ae898543fd87a4742fa76b',
 'temperature': None,
 'text': {'format': {'type': 'text'}},
 'tool_choice': 'auto',
 'tools': [],
 'top_p': None,
 'metadata': {},
 'status': 'in_progress',
 'error': None,
 'incomplete_details': None,
 'output': [],
 'usage': None}

`_ResponseStream` tracks sequence numbers, output indexes, accumulated text, and item IDs. Provider calls and persistence live outside this class.

In [ ]:
class _ResponseStream(BasicRepr):
    def __init__(self, turn):
        self.turn,self.seq,self.next_index = turn,0,0
        self.message_id,self.message_index,self.text = None,None,''
        self.reasoning_id,self.reasoning_index,self.reasoning_text,self.reasoning_ids = None,None,'',[]
        self.tool_item_ids,self.streamed_call_ids,self.used_call_ids = {},[],_history_call_ids(turn.history)

    def event(self, event_type, **kwargs):
        result = response_event(event_type, self.seq, **kwargs)
        self.seq += 1
        return result

In [ ]:
stream_state = _ResponseStream(provider_turn)
sequenced = [stream_state.event('first'), stream_state.event('second')]
test_eq([json.loads(o.split('data: ', 1)[1])['sequence_number'] for o in sequenced], [0, 1])
_event_types(sequenced)

['first', 'second']

The first text part opens an assistant item and an output-text content part. Later text yields deltas for that same item.

In [ ]:
@patch
def text_events(self:_ResponseStream, part):
    events = []
    if self.message_id is None:
        self.message_id,self.message_index = _new_id('msg'),self.next_index
        self.next_index += 1
        item = dict(id=self.message_id, type='message', status='in_progress', role='assistant', content=[])
        events.append(self.event('response.output_item.added', output_index=self.message_index, item=item))
        content = dict(type='output_text', text='', annotations=[])
        events.append(self.event('response.content_part.added', item_id=self.message_id,
            output_index=self.message_index, content_index=0, part=content))
    self.text += part.text or ''
    if part.text:
        events.append(self.event('response.output_text.delta', item_id=self.message_id,
            output_index=self.message_index, content_index=0, delta=part.text))
    return events

In [ ]:
text_stream = _ResponseStream(provider_turn)
text_frames = text_stream.text_events(Text('Hello'))
test_eq(_event_types(text_frames), ['response.output_item.added', 'response.content_part.added', 'response.output_text.delta'])
_event_types(text_frames)

['response.output_item.added',
 'response.content_part.added',
 'response.output_text.delta']

FastLLM yields a `ToolUse` after collecting its arguments. `tool_events` can therefore send item-added, argument-delta, argument-done, and item-done events without parsing provider fragments. The entire argument string fits in one delta.

In [ ]:
@patch
def tool_events(self:_ResponseStream, part):
    if not part.id or part.id in self.used_call_ids: part.id = _new_id('call')
    self.used_call_ids.add(part.id)
    self.streamed_call_ids.append(part.id)
    item_id,index,args = _new_id('fc'),self.next_index,_arguments(part.arguments)
    self.next_index += 1
    self.tool_item_ids[part.id] = item_id
    item = dict(id=item_id, type='function_call', status='in_progress', call_id=part.id, name=part.name, arguments='')
    events = [self.event('response.output_item.added', output_index=index, item=item)]
    events.append(self.event('response.function_call_arguments.delta', item_id=item_id, output_index=index, delta=args))
    events.append(self.event('response.function_call_arguments.done', item_id=item_id, output_index=index,
        name=part.name, arguments=args))
    item['status'],item['arguments'] = 'completed',args
    events.append(self.event('response.output_item.done', output_index=index, item=item))
    return events

Thinking streams as a `reasoning` output item with one summary part. The first Thinking part opens the item. Each Thinking part adds its text to the summary as a delta. The item stays open until another part arrives or the provider call ends. `close_thinking` ends it. A client can show that the model is thinking as soon as the item is added.

In [ ]:
@patch
def thinking_events(self:_ResponseStream, part):
    events = []
    if self.reasoning_id is None:
        self.reasoning_id,self.reasoning_index,self.reasoning_text = _new_id('rs'),self.next_index,''
        self.next_index += 1
        self.reasoning_ids.append(self.reasoning_id)
        events.append(self.event('response.output_item.added', output_index=self.reasoning_index, item=_reasoning_item(self.reasoning_id)))
        events.append(self.event('response.reasoning_summary_part.added', item_id=self.reasoning_id,
            output_index=self.reasoning_index, summary_index=0, part=dict(type='summary_text', text='')))
    self.reasoning_text += part.text or ''
    if part.text:
        events.append(self.event('response.reasoning_summary_text.delta', item_id=self.reasoning_id,
            output_index=self.reasoning_index, summary_index=0, delta=part.text))
    return events

@patch
def close_thinking(self:_ResponseStream):
    "Events closing the open reasoning item, if any"
    if self.reasoning_id is None: return []
    rid,text,kw = self.reasoning_id,self.reasoning_text,dict(item_id=self.reasoning_id, output_index=self.reasoning_index, summary_index=0)
    self.reasoning_id = None
    return [self.event('response.reasoning_summary_text.done', text=text, **kw),
        self.event('response.reasoning_summary_part.done', part=dict(type='summary_text', text=text), **kw),
        self.event('response.output_item.done', output_index=kw['output_index'], item=_reasoning_item(rid, text))]

In [ ]:
think_stream = _ResponseStream(provider_turn)
think_frames = think_stream.thinking_events(Thinking('Two')) + think_stream.thinking_events(Thinking(' plus two.')) + think_stream.close_thinking()
test_eq(_event_types(think_frames), ['response.output_item.added', 'response.reasoning_summary_part.added',
    'response.reasoning_summary_text.delta', 'response.reasoning_summary_text.delta', 'response.reasoning_summary_text.done',
    'response.reasoning_summary_part.done', 'response.output_item.done'])
_event_types(think_frames)

['response.output_item.added',
 'response.reasoning_summary_part.added',
 'response.reasoning_summary_text.delta',
 'response.reasoning_summary_text.delta',
 'response.reasoning_summary_text.done',
 'response.reasoning_summary_part.done',
 'response.output_item.done']

`part_events` sends thinking to `thinking_events`, text to `text_events` and client tool calls to `tool_events`. Any part other than thinking first closes an open reasoning item. This adapter doesn't emit incremental frames for other parts. `events` sends completed server calls when their step ends.

In [ ]:
@patch
def part_events(self:_ResponseStream, part):
    if isinstance(part, Thinking): return self.thinking_events(part)
    events = self.close_thinking()
    if isinstance(part, Text): events += self.text_events(part)
    elif isinstance(part, ToolUse) and not part.server: events += self.tool_events(part)
    return events

In [ ]:
tool_stream = _ResponseStream(provider_turn)
tool_frames = tool_stream.part_events(ToolUse(name='py', arguments={'code':'2+2'}))
test_eq(_event_types(tool_frames), ['response.output_item.added', 'response.function_call_arguments.delta',
    'response.function_call_arguments.done', 'response.output_item.done'])
_event_types(tool_frames)

['response.output_item.added',
 'response.function_call_arguments.delta',
 'response.function_call_arguments.done',
 'response.output_item.done']

`end_step` closes the open message item and records its ID on the Text parts that made it, with their citations as annotations. It then sends each completed server call as a `server_tool_call` item. At the end of the turn, `done_events` calls it for the parts not sent yet. `done_events` also records the streamed IDs of reasoning items and client calls on the final `Completion`. The terminal resource then names the same items as the stream.

In [ ]:
@patch
def end_step(self:_ResponseStream, parts):
    "Events ending a step: close the open message item, made from the Text parts among `parts`, then send each server call among them"
    events = []
    if self.message_id is not None:
        texts = [p for p in parts if isinstance(p, Text)]
        for p in texts: p._response_item_id = self.message_id
        content = dict(type='output_text', text=self.text, annotations=[c for p in texts for c in p.citations or []])
        item = dict(id=self.message_id, type='message', status='completed', role='assistant', content=[content])
        kw = dict(item_id=self.message_id, output_index=self.message_index, content_index=0)
        events += [self.event('response.output_text.done', text=self.text, **kw), self.event('response.content_part.done', part=content, **kw)]
        events += [self.event('response.output_text.annotation.added', annotation_index=i, annotation=c, **kw) for i,c in enumerate(content['annotations'])]
        events.append(self.event('response.output_item.done', output_index=self.message_index, item=item))
        self.message_id,self.text = None,''
    for c in parts:
        if not _is_srv(c): continue
        events.append(self.event('response.output_item.done', output_index=self.next_index, item=server_call_item(c)))
        self.next_index += 1
    return events

@patch
def done_events(self:_ResponseStream, comp):
    events = self.end_step([p for p in comp.message.content if getattr(p, '_response_item_id', None) is None])
    client_calls = [tc for tc in comp.tool_calls if not tc.server]
    for i,tc in enumerate(client_calls):
        if i < len(self.streamed_call_ids): tc.id = self.streamed_call_ids[i]
        tc._response_item_id = self.tool_item_ids.get(tc.id)
    rids = iter(self.reasoning_ids)
    for part in comp.message.content:
        if isinstance(part, Thinking): part._response_item_id = next(rids, None)
    return events

In [ ]:
terminal_comp = Completion('demo', Msg('assistant', [Text('Hello')]), 'stop')
terminal_frames = text_stream.done_events(terminal_comp)
test_eq(_event_types(terminal_frames), ['response.output_text.done', 'response.content_part.done', 'response.output_item.done'])
_event_types(terminal_frames)

['response.output_text.done',
 'response.content_part.done',
 'response.output_item.done']

`events` answers one public request as a stream of Responses events. One request can take more than one provider call. A host can pass in a stream it already started with `call`. An error from that first provider call then reaches the client before any event.

A step can make three kinds of tool call. The provider runs some itself, such as a web search, and returns them with their results already filled in. The host runs the functions in `server_tools`. The client runs the rest. `events` streams each provider call's thinking, text and client tool calls to the client as they arrive.

When the host has run every call except the provider's own, `events` asks the provider to continue under the same public response ID. First it closes the step's message item and sends the step's server calls, both the host's and the provider's. Text from the next provider call starts a new message item. A step with client calls ends the request instead: `events` returns those calls together with the host's completed `server_tool_call` items. A step with no tool calls also ends the request. The continuation state holds the original provider calls and the host's results.

The final `Completion` combines the parts of every step, in the order the stream sent them, and adds up the usage of every provider call. Before the final event, `events` calls `finalize(next_state, completion)`. The host can save the state and record billing there. Fields that `finalize` returns, such as a charged price, go into the final resource's usage.

The final response's `messages` lists every canonical output message from this request, including the host-tool rounds and the provider's replay data. A stateless client appends these to the full input of its next request.


In [ ]:
def _server_call(result):
    "A public server call with the host's completed result."
    return ToolUse(id=result.id, name=result.name, arguments=result.arguments, server=True, text=result.text)

def _visible_parts(parts, results):
    "A step's parts as the stream sends them: completed host calls become server calls, and server calls go last"
    completed = {r.id:r for r in results}
    parts = [_server_call(completed[p.id]) if isinstance(p, ToolUse) and p.id in completed else p for p in parts]
    return [p for p in parts if not _is_srv(p)] + [p for p in parts if _is_srv(p)]


In [ ]:
@patch
async def events(self:AsyncResponses, turn, finalize=None, stream=None):
    state,first = _ResponseStream(turn),turn
    yield state.event('response.created', response=pending_response(turn))
    try:
        prior,usage = [],None
        while True:
            async for part in (stream or await self.call(turn)):
                if isinstance(part, Completion): comp = part
                elif isinstance(part, Status) or (isinstance(part, ToolUse) and part.name in turn.server_tools): continue
                else:
                    for event in state.part_events(part): yield event
            for event in state.close_thinking(): yield event
            usage,stream = (comp.usage if usage is None else usage + comp.usage),None
            for tc,sid in zip([t for t in comp.tool_calls if not t.server and t.name not in turn.server_tools], state.streamed_call_ids): tc.id = sid
            results = await self.server_results(turn, comp)
            if not results or len(results) < len([t for t in comp.tool_calls if not t.server]): break
            step = _visible_parts(comp.message.content, results)
            for event in state.end_step(step): yield event
            prior += step
            state.used_call_ids.update(tc.id for tc in comp.tool_calls if not tc.server and tc.id)
            turn = self.server_turn(turn, comp, results)
        next_state = self.state(turn, comp, results)
        comp = copy.copy(comp)
        comp.message,comp.usage = Msg('assistant', prior + _visible_parts(comp.message.content, results)),usage
        for event in state.done_events(comp): yield event
        response = response_object(first.id, first.body, comp, first.previous.id if first.previous else None, first.created_at)
        response['messages'] = [msg2dict(m) for m in next_state.history[len(first.history):]]
        if finalize and (extra := await finalize(next_state, comp)): response['usage'].update(extra)
        name = 'response.incomplete' if response['status'] == 'incomplete' else 'response.completed'
        yield state.event(name, response=response)
    except asyncio.CancelledError: raise
    except ResponsesError as exc: yield state.event('error', code=exc.code, message=str(exc), param=exc.param)
    except APIError as exc: yield state.event('error', code=exc.code or 'provider_error', message=str(exc), param=None)

In [ ]:
#| hide
def step(*parts, reason='tool_calls'): return Completion('demo', Msg('assistant', list(parts)), reason, Usage())

async def run_stub(*steps):
    "The frames `events` sends for `base_body`, with host tool `lookup`, when the provider returns `steps` one per call"
    res,steps = AsyncResponses(),iter(steps)
    async def parts(comp):
        for p in comp.message.content + [comp]: yield p
    async def call(turn): return parts(next(steps))
    res.call = call
    return [f async for f in res.events(res.prepare(base_body, server_tools=[lookup]))]

The next two examples replace the provider with a stub, `run_stub`, that returns one prepared step per provider call. Each example checks the final event for the answer. In the normal case, the model calls the host's `lookup`. `events` runs it, asks the provider to continue, and ends with the answer:

In [ ]:
lookup_call = ToolUse(id='call_1', name='lookup', arguments=dict(word='gorp'))
answer = step(Text('Gorp is trail mix.'), reason='stop')
assert 'Gorp is trail mix.' in (await run_stub(step(lookup_call), answer))[-1]
dup_frames = await run_stub(step(ToolUse(id='call_1', name='py', arguments=dict(code='1')), ToolUse(id='call_1', name='py', arguments=dict(code='2'))))
s_ids = [json.loads(f.split('data: ', 1)[1])['item']['call_id'] for f in dup_frames if 'response.output_item.done' in f]
done_ids = [o['call_id'] for o in json.loads(dup_frames[-1].split('data: ', 1)[1])['response']['output'] if o['type']=='function_call']
test_eq((len(set(s_ids)), s_ids), (2, done_ids))

A provider can run a tool itself, such as Claude Code's `WebFetch`, in the same step as a host tool. `server_results` returns nothing for the provider's call, which already has its result. `events` still runs `lookup` and continues:

In [ ]:
web_fetch = ToolUse(id='srv_1', name='WebFetch', arguments=dict(url='https://example.com/gorp'), server=True, text='Gorp: trail mix')
assert 'Gorp is trail mix.' in (await run_stub(step(web_fetch, lookup_call), answer))[-1]

A host that streams `events` directly ties the provider loop to the HTTP consumer. If a disconnect cancels that loop before completion, `finalize` doesn't run and final usage may be unavailable.

`serve` runs the loop in a separate task, retained in `tasks`, and sends its frames through a queue. Closing the returned iterator stops delivery, not the provider task. The task can finish and report usage through `finalize`. An optional `timeout` bounds its lifetime and produces an error frame on expiry.

In [ ]:
@patch
def serve(self:AsyncResponses, turn, finalize=None, stream=None, timeout=None):
    "The frames of `events`, from a task that outlives its reader, so a disconnected client still gets its turn finalized"
    q,closed = asyncio.Queue(),False
    async def _run():
        try:
            async with asyncio.timeout(timeout):
                async for frame in self.events(turn, finalize=finalize, stream=stream):
                    if not closed: q.put_nowait(frame)
        except TimeoutError:
            if not closed: q.put_nowait(response_event('error', 0, code='timeout', message=f'turn exceeded {timeout}s', param=None))
        finally:
            if not closed: q.put_nowait(None)
    t = asyncio.create_task(_run())
    self.tasks.add(t)
    t.add_done_callback(self.tasks.discard)
    async def _frames():
        nonlocal closed
        try:
            while (frame := await q.get()) is not None: yield frame
        finally:
            closed = True
            while not q.empty(): q.get_nowait()
    return _frames()

## Provider turns

These examples use GPT-5.6 Luna through Codex and OpenAI to exercise the client-tool and host-tool paths. Cachy records HTTP responses for repeatable notebook runs. Production code doesn't enable it.

In [ ]:
enable_cachy(doms=vendor_doms)


In [ ]:
#| hide
defaults.reasoning_effort = 'l'

In [ ]:
py_tool = dict(type='function', name='py', description='Run Python code',
    parameters=dict(type='object', properties={'code':{'type':'string'}}, required=['code'], additionalProperties=False))
live_body = dict(model=f'codex/{luna}', input='Use py to calculate 12345*6789. Do not calculate it yourself.',
    instructions='Return one py tool call.', tools=[py_tool], tool_choice='required', parallel_tool_calls=True)
live_body

{'model': 'codex/gpt-6-luna',
 'input': 'Use py to calculate 12345*6789. Do not calculate it yourself.',
 'instructions': 'Return one py tool call.',
 'tools': [{'type': 'function',
   'name': 'py',
   'description': 'Run Python code',
   'parameters': {'type': 'object',
    'properties': {'code': {'type': 'string'}},
    'required': ['code'],
    'additionalProperties': False}}],
 'tool_choice': 'required',
 'parallel_tool_calls': True}

In [ ]:
#| hide
live_final = []
async def capture_live(state, comp): live_final.append((state, comp))

The client-tool request ends with `response.completed` and a call for the client to run. Its finalizer retains the next state. The test checks that serialization preserves the provider's raw message.

In [ ]:
live_turn = responses.prepare(live_body)
live_frames = [frame async for frame in responses.events(live_turn, finalize=capture_live)]
live_state,live_comp = live_final[0]
test_eq((live_comp.finish_reason, live_comp.tool_calls[0].name), ('tool_calls', 'py'))
test_eq(live_frames[-1].splitlines()[0], 'event: response.completed')
test_eq(ResponseState.from_dict(live_state.to_dict()).history[-1].raw, live_state.history[-1].raw)
live_comp.tool_calls

[ToolUse(raw={'type': 'function_call', 'id': 'fc_01ccbee287bbc822016b038a84269087d0bac1098a1939ce35', 'status': 'completed'}, cache_control=None, id='call_e6e30f825ba84001b3786e2aa6ad18a4', name='py', arguments={'code': '12345 * 6789'}, server=False, text=None, md=False, full=False)]

The client sends back a tool output and the previous public response ID. FastLLM can use an upstream continuation ID when available, but always creates a fresh public resource for this new request.

In [ ]:
live_call = live_comp.tool_calls[0]
continued_body = dict(model=live_body['model'], tools=[py_tool], tool_choice='none', previous_response_id=live_state.id)
continued_body['input'] = [dict(type='function_call_output', call_id=live_call.id, output='83810205')]
continued_body

{'model': 'codex/gpt-6-luna',
 'tools': [{'type': 'function',
   'name': 'py',
   'description': 'Run Python code',
   'parameters': {'type': 'object',
    'properties': {'code': {'type': 'string'}},
    'required': ['code'],
    'additionalProperties': False}}],
 'tool_choice': 'none',
 'previous_response_id': 'resp_8431e3fe7f4b3b4b13d96c6610f6965d',
 'input': [{'type': 'function_call_output',
   'call_id': 'call_e6e30f825ba84001b3786e2aa6ad18a4',
   'output': '83810205'}]}

In [ ]:
#| hide
continued_final = []
async def capture_continued(state, comp): continued_final.append((state, comp))

Codex has no usable provider continuation ID, so this continuation replays the saved messages. The public `previous_response_id` still works.

A `function_call_output` can omit its function name. `prepare` finds that name on the matching call in history; adapters such as Gemini need it when sending the result.

In [ ]:
continued_turn = responses.prepare(continued_body, live_state)
test_eq(first(p for m in continued_turn.provider_messages for p in m.content if isinstance(p, ToolResult)).name, 'py')
continued_frames = [frame async for frame in responses.events(continued_turn, finalize=capture_continued)]
continued_state,continued_comp = continued_final[0]
assert '83810205' in continued_comp.message.text.replace(',', '')
test_eq(continued_turn.previous.id, live_state.id)
continued_comp.message

**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

83,810,205

::: details

- raw: `None`
- citations: `[]`

:::

</contents>

Here the host supplies `lookup` and no client tools. FastLLM runs the lookup, gives its result to the provider, and returns the answer in one public response. The stream has no client function call to execute. The saved history contains the tool round, and usage includes both provider calls.

In [ ]:
lookup_final = []
async def capture_lookup(state, comp):
    lookup_final.append((state, comp))
    return dict(cost=0.0123)
lookup_body = dict(model=f'openai/{luna}', input="What does 'gorp' mean according to the glossary? Use lookup, then answer in one sentence.", tools=[])
lookup_frames = [frame async for frame in responses.events(responses.prepare(lookup_body, server_tools=[lookup]), finalize=capture_lookup)]
lookup_state,lookup_comp = lookup_final[0]
lookup_comp.message.text

'“Gorp” means a trail mix of nuts and dried fruit.'

The finalizer receives the combined completion and the next state. The client sees a completed `server_tool_call` with the lookup result in `output`, before the answer's text. The terminal usage also includes the example price returned by the finalizer.


In [ ]:
test_eq([m.role for m in lookup_state.history], ['user', 'assistant', 'tool', 'assistant'])
done = json.loads(lookup_frames[-1].split('data: ', 1)[1])['response']
assert 'function_call' not in ''.join(lookup_frames[:-1])
assert all(o['type'] != 'function_call' for o in done['output'])
assert '"server_tool_call"' in ''.join(lookup_frames)
srv_at = first(i for i,f in enumerate(lookup_frames) if '"server_tool_call"' in f)
assert srv_at < max(i for i,t in enumerate(_event_types(lookup_frames)) if t == 'response.output_text.delta')
assert '"cost":0.0123' in lookup_frames[-1]
assert 'trail mix' in lookup_comp.message.text.lower(), lookup_comp.message.text
lookup_frames[-1]

'event: response.completed\ndata: {"type":"response.completed","response":{"id":"resp_eabdffbbdc2d7c16b4e4ebf0860e456c","object":"response","created_at":1795401222,"instructions":null,"model":"openai/gpt-6-luna","parallel_tool_calls":true,"previous_response_id":null,"reasoning":null,"store":true,"prompt_cache_key":"34e42263abdb8695f5406dd826ec8470","temperature":null,"text":{"format":{"type":"text"}},"tool_choice":"auto","tools":[],"top_p":null,"metadata":{},"completed_at":1795401222,"status":"completed","error":null,"incomplete_details":null,"output":[{"id":"srv_4589ab3e55964098fcb967f45f4f8212","status":"completed","type":"server_tool_call","name":"lookup","arguments":"{\\"word\\":\\"gorp\\"}","output":"a trail mix of nuts and dried fruit"},{"id":"msg_f6701ccff6e1ca95aad92c9a8d7fa10c","type":"message","status":"completed","role":"assistant","content":[{"type":"output_text","text":"\\u201cGorp\\u201d means a trail mix of nuts and dried fruit.","annotations":[]}]}],"usage":{"input_toke

This consumer takes two frames from `serve` and closes, simulating a disconnect. The test waits for the provider task and checks that the finalizer ran exactly once with the full turn's usage.

In [ ]:
served_final = []
async def capture_served(state, comp): served_final.append((state, comp))
frames = responses.serve(responses.prepare(live_body), finalize=capture_served)
first_two = [await anext(frames), await anext(frames)]
await frames.aclose()
await asyncio.gather(*responses.tasks)
test_eq(len(served_final), 1)
assert served_final[0][1].usage.total_tokens > 0
first_two[0].splitlines()[0]

'event: response.created'

## Export

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()